# Ubiquitin (BMRB 11547_1): SLAE-CS ¹⁵N error in β5 for different input structures (Fig. S22b)

| input | what it is |
|---|---|
| `vanilla_AF2` | the PANAV test structure (ColabFold, no custom template, rank_001 = AF2 model_4) |
| `AF2+NMR_msa` | ColabFold model_1 with NMR model 1 (PDB 2RU6) as the only template, full MSA |
| `AF2+NMR_single` | same, single-sequence mode |
| `NMR_m01` … `NMR_m10` | the 10 NMR models of 2RU6, predicted one by one |

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "data"
pd.set_option("display.precision", 3)
G = json.load(open(f"{DATA}/gt_11547_1.json"))
D = json.load(open(f"{DATA}/predictions.json"))
seq = G["sequence"]; L = len(seq)
gt = np.array([np.nan if v is None else v for v in G["N"]], float)
print(f"{G['pid']}: {L} residues\n{seq}")
print("inputs:", list(D["pred"]))

## 1. Predictions per input and the residue mask

In [ ]:
seedmean = {k: np.mean(np.array(v, float), axis=0) for k, v in D["pred"].items()}
nmr_keys = [k for k in seedmean if k.startswith("NMR_m")]
seedmean["NMR_10models"] = np.mean([seedmean[k] for k in nmr_keys], axis=0)

pos = np.all([np.array(D["mask"][k], bool) for k in D["mask"]], axis=0)
mask = pos & np.array(G["flag_keep"]) & np.array(G["tag_keep"]) & np.array(G["gt_valid"]) & ~np.isnan(gt)
OUTLIERS = [9, 20, 46]                                   # 1-based
mask_clean = mask.copy(); mask_clean[[i - 1 for i in OUTLIERS]] = False
print(f"flagged: {int((~np.array(G['flag_keep'])).sum())}, tag-masked: {int((~np.array(G['tag_keep'])).sum())}, "
      f"no GT: {int((~np.array(G['gt_valid'])).sum())}  ->  scored {mask.sum()} residues ({mask_clean.sum()} without outliers)")

In [ ]:
# ---- b5 (68-72) MAE: vanilla AF2 vs AF2 + NMR template (single seq / MSA) ----
def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base + 2, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })
set_style(base=14)

B5_INPUTS = ["vanilla_AF2", "AF2+NMR_msa", "AF2+NMR_single"]
B5_LABELS = ["AF2\n(vanilla)", "AF2 + NMR str tmpl\n(MSA)", "AF2 + NMR str tmpl\n(single seq)"]
B5_COLORS = ["#999999", "#98C379", "#61AFEF"]

s, e = 68, 72
sel = mask_clean.copy(); sel[:s - 1] = False; sel[e:] = False
res_lbl = [f"{i + 1}{seq[i]}" for i in np.where(sel)[0]]
abs_err = {k: np.abs(seedmean[k][sel] - gt[sel]) for k in B5_INPUTS}      # per-residue |error| of the seed mean
mae = np.array([abs_err[k].mean() for k in B5_INPUTS])              

ERR_MODE = "sem"     # "sem": SEM over residues | "sd": SD over residues | "seed": SD of per-seed MAE over the 10 seeds
if ERR_MODE == "seed":
    seed_mae = {k: np.abs(np.array(D["pred"][k], float)[:, sel] - gt[sel]).mean(axis=1) for k in B5_INPUTS}
    err = np.array([seed_mae[k].std(ddof=1) for k in B5_INPUTS])
    ERR_LABEL = f"SD over {len(D['pred'][B5_INPUTS[0]])} seeds"
else:
    sd = np.array([abs_err[k].std(ddof=1) for k in B5_INPUTS])
    err = sd / np.sqrt(sel.sum()) if ERR_MODE == "sem" else sd
    ERR_LABEL = f"{ERR_MODE.upper()} over {sel.sum()} residues"


fig, ax = plt.subplots(figsize=(6, 5.5))
x = np.arange(len(B5_INPUTS))
ax.bar(x, mae, width=0.8, color=B5_COLORS, edgecolor="white", zorder=2)
ax.errorbar(x, mae, yerr=err, fmt="none", ecolor="k", elinewidth=1.4, capsize=6, capthick=1.4, zorder=4)
rng = np.random.default_rng(0)
for i, k in enumerate(B5_INPUTS):                                         # individual residues on top
    ax.scatter(x[i] + rng.uniform(-0.12, 0.12, sel.sum()), abs_err[k], s=28, color="k",
               alpha=0.6, edgecolors="none", zorder=3)
ymax = max(max(v.max() for v in abs_err.values()), (mae + err).max())
for i, m in enumerate(mae):
    ax.text(i, ymax * 1.04, f"{m:.2f}", ha="center", va="bottom")
ax.set_xticks(x); ax.set_xticklabels(B5_LABELS)
ax.set_ylim(0, ymax * 1.15)


ax.set_ylabel("$\\mathregular{^{15}}$N MAE (ppm)")
ax.set_title(f"β5 residues {s}-{e}  (n = {sel.sum()})")
# ax.grid(axis="y", alpha=0.3, ls="--"); [ax.spines[sp].set_visible(False) for sp in ("top", "right")]
plt.tight_layout()
os.makedirs("Figs", exist_ok=True)
fig.savefig("Figs/11547_1_b5_MAE_af2_template.svg", bbox_inches="tight")
fig.savefig("Figs/11547_1_b5_MAE_af2_template.png", dpi=300, bbox_inches="tight")
plt.show()
print(pd.DataFrame(abs_err, index=res_lbl).round(2))
